<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/MakineOgrenmesi_Vize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sayın Hocam,

Ödevi incelerken aşağıdaki hususları dikkate almanızı rica ederim.

Kaggle veri indirme ve yükleme süreçlerinde, veri boyutunun yüksek olması nedeniyle doğrudan klasör olarak indirme yapılamamıştır. Bu nedenle veri, Kaggle API kullanılarak .json formatında indirilmiş ve Colab ortamına yüklenmiştir. Colab üzerinde veri erişimi için ilgili kaggle.json dosyasının yolu tanımlanarak API yetkilendirmesi gerçekleştirilmiştir. Uygulamanın çalıştırılabilmesi için önceden KAGGLE_USERNAME ve KAGGLE_KEY değişkenlerinin tanımlanması gerekmektedir.

Bilgilerinize sunarım.

Saygılarımla

Soru 1- Veri Seti ve Sınıf Seçimi

In [ ]:
from google.colab import userdata
import os

print("Colab Secrets'tan Kaggle API bilgileri okunuyor...")
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("API bilgileri başarıyla okundu.")
except userdata.SecretNotFoundError:
    print("\nHata: 'KAGGLE_USERNAME' veya 'KAGGLE_KEY' gizli anahtarları bulunamadı.")
    print("Lütfen yukarıdaki adımları takip ederek anahtarları doğru şekilde eklediğinizden emin olun.")
except Exception as e:
    print(f"\nBeklenmeyen bir hata oluştu: {e}")


In [ ]:
!pip install -q kaggle

In [ ]:
!kaggle datasets download -d mragpavank/breast-cancer --force

In [ ]:
import pandas as pd
import zipfile
import os
if 'df' not in locals() and 'df' not in globals():
    zip_file_path = '/content/breast-cancer.zip'
    extracted_dir = '/content/'
    df = None

    if not os.path.exists(zip_file_path):
        print(f"Hata: '{zip_file_path}' dosyası bulunamadı. Lütfen 'kaggle datasets download' adımını kontrol edin.")
    else:
        try:
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extracted_dir)
            print(f"'{zip_file_path}' başarıyla '{extracted_dir}' konumuna çıkarıldı.")

            csv_file_name = None
            for root, dirs, files in os.walk(extracted_dir):
                for file in files:
                    if file.endswith('.csv'):
                        csv_file_name = os.path.join(root, file)
                        break
                if csv_file_name:
                    break

            if csv_file_name:
                df = pd.read_csv(csv_file_name)
                print(f"'{csv_file_name}' başarıyla yüklendi.")
            else:
                print(f"Hata: '{zip_file_path}' içinden bir CSV dosyası bulunamadı. Lütfen zip içeriğini kontrol edin.")

        except Exception as e:
            print(f"Zip dosyasını çıkarırken veya CSV dosyasını bulurken bir hata oluştu: {e}")
else:
    print("df DataFrame'i zaten tanımlı.")

if df is not None:
    X = df.drop(columns=['id', 'diagnosis'])
    y = df['diagnosis']

    print("X (özellikler) Veri Çerçevesinin İlk 5 Satırı:")
    display(X.head())

    print("\ny (hedef) Serisinin İlk 5 Satırı:")
    display(y.head())
else:
    print("df DataFrame'i yüklenemediği için X ve y oluşturulamadı. Lütfen yukarıdaki hataları giderin.")

Soru-2 Veri Seti Kalite Kontrolleri

In [ ]:
print("df.isnull().sum() ile eksik değer kontrolü:")
display(df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("X veri çerçevesindeki sayısal özellikler için Kutu Grafikleri (Boxplots): İlk 5 sütun gösteriliyor.")

numerical_cols = X.select_dtypes(include=['number']).columns

for col in numerical_cols[:5]:
    plt.figure(figsize=(8, 6))
    sns.boxplot(y=X[col])
    plt.title(f'{col} İçin Kutu Grafiği')
    plt.ylabel(col)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
print("\nDataFrame'deki her sütunun bilgi özeti (df.info()):\n")
df.info()

In [ ]:
print("\nVeri Tipleri ve Dağılım İncelemesi:\n")
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns
categorical_features = df.select_dtypes(include=['object', 'category']).columns

print(f"Sayısal Değişken Sayısı: {len(numerical_features)}")
print(f"Sayısal Değişkenler: {list(numerical_features)}")
print(f"\nKategorik Değişken Sayısı: {len(categorical_features)}")
print(f"Kategorik Değişkenler: {list(categorical_features)}")

print("\nHer bir sayısal değişkenin temel istatistikleri (describe()):")
display(df[numerical_features].describe().T)

print("\nHer bir kategorik değişkenin değer sayıları (value_counts()):")
for col in categorical_features:
    print(f"\n'{col}' değişkeninin değer sayıları:")
    display(df[col].value_counts())

Soru-3 Keşifsel Veri Analizi (EDA)

In [ ]:
print("Sayısal özellikler için İstatistiksel Özellikler (mean, std, min, Q1, median (Q2), Q3, max):")
display(df[numerical_features].describe().T)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print("Pearson Korelasyon Matrisi:")
correlation_matrix = df[numerical_features].corr(method='pearson')
display(correlation_matrix.round(2))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print("\nKorelasyon Matrisinin Isı Haritası (Heatmap):")
plt.figure(figsize=(20, 18))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Pearson Korelasyon Matrisi', fontsize=20)
plt.show()

Soru-4 Veri Ölçeklendirme (Scaling)

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("X_scaled (Ölçeklendirilmiş Özellikler) Veri Çerçevesinin İlk 5 Satırı:")
display(X_scaled.head())

print("\nX_scaled verisinin ortalamaları (yaklaşık 0 olmalı):")
print(X_scaled.mean())

print("\nX_scaled verisinin standart sapmaları (yaklaşık 1 olmalı):")
print(X_scaled.std())

Soru-5 Veri Setinin Bölünmesi

In [ ]:
from sklearn.model_selection import train_test_split

# İlk olarak veriyi eğitim (%70) ve geçici (%30) setler
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=(2/3), random_state=42, stratify=y_temp)

print(f"Eğitim Seti Boyutu (X_train): {X_train.shape}")
print(f"Eğitim Seti Boyutu (y_train): {y_train.shape}")
print(f"Doğrulama Seti Boyutu (X_val): {X_val.shape}")
print(f"Doğrulama Seti Boyutu (y_val): {y_val.shape}")
print(f"Test Seti Boyutu (X_test): {X_test.shape}")
print(f"Test Seti Boyutu (y_test): {y_test.shape}")

print("\nSetlerdeki sınıf dağılımları (stratify kontrolü):")
print("y_train dağılımı:\n", y_train.value_counts(normalize=True))
print("y_val dağılımı:\n", y_val.value_counts(normalize=True))
print("y_test dağılımı:\n", y_test.value_counts(normalize=True))

Soru-6 Özellik Seçimi ve Boyut İndirgeme

In [ ]:
# Ham veri temsili olarak X_scaled'ı kullanıyoruz
X_raw_representation = X_scaled.copy()

print("Ham Veri Temsilinin (X_raw_representation) İlk 5 Satırı:")
display(X_raw_representation.head())

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=None)
pca.fit(X_scaled.dropna(axis=1))

explained_variance_ratios = pca.explained_variance_ratio_

# Ortalama açıklanan varyans oranını hesapla
average_explained_variance_ratio = np.mean(explained_variance_ratios)
print(f"Ortalama Açıklanan Varyans Oranı: {average_explained_variance_ratio:.4f}")

# Ortalamadan büyük olan bileşen sayısını belirle
num_components = np.sum(explained_variance_ratios > average_explained_variance_ratio)
print(f"Açıklanan varyans oranı ortalamadan büyük olan bileşen sayısı: {num_components}")

# Belirlenen bileşen sayısı ile PCA'yı yeniden uygula
pca_final = PCA(n_components=num_components)
X_pca = pca_final.fit_transform(X_scaled.dropna(axis=1))

print("\nPCA ile boyut indirgenmiş veri setinin boyutu:")
print(X_pca.shape)

X_pca_df = pd.DataFrame(X_pca, columns=[f'PC_{i+1}' for i in range(num_components)])
print("\nPCA ile oluşturulan ilk 5 bileşen:")
display(X_pca_df.head())

In [ ]:
# Explained variance grafiğini çizme
plt.figure(figsize=(10, 6))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.xlabel('Bileşen Sayısı')
plt.ylabel('Kümülatif Açıklanan Varyans Oranı')
plt.title('Bileşen Sayısına Göre Kümülatif Açıklanan Varyans')
plt.grid(True)
plt.axvline(x=num_components, color='r', linestyle=':', label=f'{num_components} Bileşen (Ortalama Üzeri)')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x=[f'PC_{i+1}' for i in range(len(explained_variance_ratios))],
            y=explained_variance_ratios,
            color='skyblue')
plt.axhline(y=average_explained_variance_ratio, color='red', linestyle='--', label='Ortalama Açıklanan Varyans Oranı')
plt.xlabel('Temel Bileşenler')
plt.ylabel('Açıklanan Varyans Oranı')
plt.title('Her Bir Temel Bileşenin Açıklanan Varyans Oranı')
plt.xticks(rotation=90)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import matplotlib.pyplot as plt
import seaborn as sns

# LDA modelini oluşturma
lda = LinearDiscriminantAnalysis(n_components=1)

# LDA'yı "X_scaled" verisine uygula
X_lda = lda.fit_transform(X_scaled.dropna(axis=1), y)

print("LDA ile boyut indirgenmiş veri setinin boyutu:")
print(X_lda.shape)

# LDA ile oluşturulan veriyi bir DataFrame'e dönüştür
X_lda_df = pd.DataFrame(X_lda, columns=[f'LDA_{i+1}' for i in range(X_lda.shape[1])])

# Sınıf etiketlerini de ekleyerek görselleştirme için hazır hale getirin
X_lda_df['diagnosis'] = y.reset_index(drop=True)

print("\nLDA ile oluşturulan ilk 5 bileşen ve hedef değişken:")
display(X_lda_df.head())

In [ ]:
print("\nİlk LDA bileşeniyle sınıflar arası ayrım (1D Dağılım Grafiği):")

plt.figure(figsize=(10, 6))
sns.histplot(data=X_lda_df, x='LDA_1', hue='diagnosis', kde=True, palette='viridis', bins=30)
plt.title('İlk LDA Bileşeni ile Sınıflandırma (Yoğunluk Dağılımı)', fontsize=16)
plt.xlabel('LDA Bileşeni 1', fontsize=12)
plt.ylabel('Yoğunluk / Frekans', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Diagnosis')
plt.show()

Soru-7 Makine Öğrenmesi Modellerinin Kurulması

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

models = {}
results = {}

In [ ]:
# Lojistik Regresyon modelini oluşturun ve eğit
log_reg_raw = LogisticRegression(random_state=42)

# Bu sütun, orijinal veri setinde tamamen boş olduğu için model eğitiminden çıkar
log_reg_raw.fit(X_train.drop(columns=['Unnamed: 32']), y_train)

models['LogisticRegression_Raw'] = log_reg_raw

y_pred_raw = log_reg_raw.predict(X_val.drop(columns=['Unnamed: 32']))

# Performans metriklerini hesapla
accuracy_raw = accuracy_score(y_val, y_pred_raw)
precision_raw = precision_score(y_val, y_pred_raw, pos_label='M') # 'M' malign için pozitif sınıf
recall_raw = recall_score(y_val, y_pred_raw, pos_label='M')
f1_raw = f1_score(y_val, y_pred_raw, pos_label='M')

# Sonuçları saklayın
results['LogisticRegression_Raw'] = {
    'Accuracy': accuracy_raw,
    'Precision': precision_raw,
    'Recall': recall_raw,
    'F1-Score': f1_raw
}

print("Ham Veri Üzerinde Lojistik Regresyon Performansı (Doğrulama Seti):")
for metric, value in results['LogisticRegression_Raw'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# PCA dönüşümünü eğitim ve doğrulama setlerine uygula

X_train_pca = pca_final.transform(X_train.dropna(axis=1))
X_val_pca = pca_final.transform(X_val.dropna(axis=1))

# Lojistik Regresyon modelini PCA verisi üzerinde oluştur ve eğit
log_reg_pca = LogisticRegression(random_state=42)
log_reg_pca.fit(X_train_pca, y_train)

models['LogisticRegression_PCA'] = log_reg_pca

# Doğrulama seti üzerinde tahminler yap
y_pred_pca = log_reg_pca.predict(X_val_pca)

# Performans metriklerini hesapla
accuracy_pca = accuracy_score(y_val, y_pred_pca)
precision_pca = precision_score(y_val, y_pred_pca, pos_label='M')
recall_pca = recall_score(y_val, y_pred_pca, pos_label='M')
f1_pca = f1_score(y_val, y_pred_pca, pos_label='M')

results['LogisticRegression_PCA'] = {
    'Accuracy': accuracy_pca,
    'Precision': precision_pca,
    'Recall': recall_pca,
    'F1-Score': f1_pca
}

print("PCA Verisi Üzerinde Lojistik Regresyon Performansı (Doğrulama Seti):")
for metric, value in results['LogisticRegression_PCA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# LDA dönüşümünü eğitim ve doğrulama setlerine uygula

X_train_lda = lda.transform(X_train.dropna(axis=1))
X_val_lda = lda.transform(X_val.dropna(axis=1))

# Lojistik Regresyon modelini LDA verisi üzerinde oluştur ve eğit
log_reg_lda = LogisticRegression(random_state=42)
log_reg_lda.fit(X_train_lda, y_train)

models['LogisticRegression_LDA'] = log_reg_lda

# Doğrulama seti üzerinde tahminler yap
y_pred_lda = log_reg_lda.predict(X_val_lda)

# Performans metriklerini hesapla
accuracy_lda = accuracy_score(y_val, y_pred_lda)
precision_lda = precision_score(y_val, y_pred_lda, pos_label='M')
recall_lda = recall_score(y_val, y_pred_lda, pos_label='M')
f1_lda = f1_score(y_val, y_pred_lda, pos_label='M')

results['LogisticRegression_LDA'] = {
    'Accuracy': accuracy_lda,
    'Precision': precision_lda,
    'Recall': recall_lda,
    'F1-Score': f1_lda
}

print("LDA Verisi Üzerinde Lojistik Regresyon Performansı (Doğrulama Seti):")
for metric, value in results['LogisticRegression_LDA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Karar Ağacı modelini oluşturun ve eğit
dec_tree_raw = DecisionTreeClassifier(random_state=42)

# NaN içeren 'Unnamed: 32' sütununu düşürerek modeli eğit
dec_tree_raw.fit(X_train.drop(columns=['Unnamed: 32']), y_train)

models['DecisionTreeClassifier_Raw'] = dec_tree_raw

y_pred_dec_tree_raw = dec_tree_raw.predict(X_val.drop(columns=['Unnamed: 32']))

# Performans metriklerini hesapla
accuracy_dec_tree_raw = accuracy_score(y_val, y_pred_dec_tree_raw)
precision_dec_tree_raw = precision_score(y_val, y_pred_dec_tree_raw, pos_label='M')
recall_dec_tree_raw = recall_score(y_val, y_pred_dec_tree_raw, pos_label='M')
f1_dec_tree_raw = f1_score(y_val, y_pred_dec_tree_raw, pos_label='M')

results['DecisionTreeClassifier_Raw'] = {
    'Accuracy': accuracy_dec_tree_raw,
    'Precision': precision_dec_tree_raw,
    'Recall': recall_dec_tree_raw,
    'F1-Score': f1_dec_tree_raw
}

print("Ham Veri Üzerinde Karar Ağacı Performansı (Doğrulama Seti):")
for metric, value in results['DecisionTreeClassifier_Raw'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Karar Ağacı modelini PCA verisi üzerinde oluştur ve eğit
dec_tree_pca = DecisionTreeClassifier(random_state=42)
dec_tree_pca.fit(X_train_pca, y_train)

models['DecisionTreeClassifier_PCA'] = dec_tree_pca

# Doğrulama seti üzerinde tahminler yap
y_pred_dec_tree_pca = dec_tree_pca.predict(X_val_pca)

# Performans metriklerini hesapla
accuracy_dec_tree_pca = accuracy_score(y_val, y_pred_dec_tree_pca)
precision_dec_tree_pca = precision_score(y_val, y_pred_dec_tree_pca, pos_label='M')
recall_dec_tree_pca = recall_score(y_val, y_pred_dec_tree_pca, pos_label='M')
f1_dec_tree_pca = f1_score(y_val, y_pred_dec_tree_pca, pos_label='M')

results['DecisionTreeClassifier_PCA'] = {
    'Accuracy': accuracy_dec_tree_pca,
    'Precision': precision_dec_tree_pca,
    'Recall': recall_dec_tree_pca,
    'F1-Score': f1_dec_tree_pca
}

print("PCA Verisi Üzerinde Karar Ağacı Performansı (Doğrulama Seti):")
for metric, value in results['DecisionTreeClassifier_PCA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Karar Ağacı modelini LDA verisi üzerinde oluştur ve eğit
dec_tree_lda = DecisionTreeClassifier(random_state=42)
dec_tree_lda.fit(X_train_lda, y_train)

models['DecisionTreeClassifier_LDA'] = dec_tree_lda

# Doğrulama seti üzerinde tahminler yap
y_pred_dec_tree_lda = dec_tree_lda.predict(X_val_lda)

# Performans metriklerini hesapla
accuracy_dec_tree_lda = accuracy_score(y_val, y_pred_dec_tree_lda)
precision_dec_tree_lda = precision_score(y_val, y_pred_dec_tree_lda, pos_label='M')
recall_dec_tree_lda = recall_score(y_val, y_pred_dec_tree_lda, pos_label='M')
f1_dec_tree_lda = f1_score(y_val, y_pred_dec_tree_lda, pos_label='M')

results['DecisionTreeClassifier_LDA'] = {
    'Accuracy': accuracy_dec_tree_lda,
    'Precision': precision_dec_tree_lda,
    'Recall': recall_dec_tree_lda,
    'F1-Score': f1_dec_tree_lda
}

print("LDA Verisi Üzerinde Karar Ağacı Performansı (Doğrulama Seti):")
for metric, value in results['DecisionTreeClassifier_LDA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore') # Suppress warnings

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest modelini oluştur ve eğit
rf_raw = RandomForestClassifier(random_state=42)

rf_raw.fit(X_train.drop(columns=['Unnamed: 32']), y_train)

models['RandomForest_Raw'] = rf_raw

y_pred_rf_raw = rf_raw.predict(X_val.drop(columns=['Unnamed: 32']))

# Performans metriklerini hesapla
accuracy_rf_raw = accuracy_score(y_val, y_pred_rf_raw)
precision_rf_raw = precision_score(y_val, y_pred_rf_raw, pos_label='M')
recall_rf_raw = recall_score(y_val, y_pred_rf_raw, pos_label='M')
f1_rf_raw = f1_score(y_val, y_pred_rf_raw, pos_label='M')

results['RandomForest_Raw'] = {
    'Accuracy': accuracy_rf_raw,
    'Precision': precision_rf_raw,
    'Recall': recall_rf_raw,
    'F1-Score': f1_rf_raw
}

print("Ham Veri Üzerinde Random Forest Performansı (Doğrulama Seti):")
for metric, value in results['RandomForest_Raw'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Random Forest modelini PCA verisi üzerinde oluştur ve eğit
rf_pca = RandomForestClassifier(random_state=42)
rf_pca.fit(X_train_pca, y_train)

models['RandomForest_PCA'] = rf_pca

y_pred_rf_pca = rf_pca.predict(X_val_pca)

accuracy_rf_pca = accuracy_score(y_val, y_pred_rf_pca)
precision_rf_pca = precision_score(y_val, y_pred_rf_pca, pos_label='M')
recall_rf_pca = recall_score(y_val, y_pred_rf_pca, pos_label='M')
f1_rf_pca = f1_score(y_val, y_pred_rf_pca, pos_label='M')

results['RandomForest_PCA'] = {
    'Accuracy': accuracy_rf_pca,
    'Precision': precision_rf_pca,
    'Recall': recall_rf_pca,
    'F1-Score': f1_rf_pca
}

print("PCA Verisi Üzerinde Random Forest Performansı (Doğrulama Seti):")
for metric, value in results['RandomForest_PCA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Random Forest modelini LDA verisi üzerinde oluştur ve eğit
rf_lda = RandomForestClassifier(random_state=42)
rf_lda.fit(X_train_lda, y_train)

models['RandomForest_LDA'] = rf_lda

y_pred_rf_lda = rf_lda.predict(X_val_lda)

accuracy_rf_lda = accuracy_score(y_val, y_pred_rf_lda)
precision_rf_lda = precision_score(y_val, y_pred_rf_lda, pos_label='M')
recall_rf_lda = recall_score(y_val, y_pred_rf_lda, pos_label='M')
f1_rf_lda = f1_score(y_val, y_pred_rf_lda, pos_label='M')

results['RandomForest_LDA'] = {
    'Accuracy': accuracy_rf_lda,
    'Precision': precision_rf_lda,
    'Recall': recall_rf_lda,
    'F1-Score': f1_rf_lda
}

print("LDA Verisi Üzerinde Random Forest Performansı (Doğrulama Seti):")
for metric, value in results['RandomForest_LDA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_val_encoded = le.transform(y_val)

pos_label_encoded = le.transform(['M'])[0]

# XGBoost modelini oluştur ve eğit
xgb_raw = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# NaN içeren 'Unnamed: 32' sütununu düşürerek modeli eğit
xgb_raw.fit(X_train.drop(columns=['Unnamed: 32']), y_train_encoded)

models['XGBoost_Raw'] = xgb_raw

y_pred_xgb_raw = xgb_raw.predict(X_val.drop(columns=['Unnamed: 32']))

accuracy_xgb_raw = accuracy_score(y_val_encoded, y_pred_xgb_raw)
precision_xgb_raw = precision_score(y_val_encoded, y_pred_xgb_raw, pos_label=pos_label_encoded)
recall_xgb_raw = recall_score(y_val_encoded, y_pred_xgb_raw, pos_label=pos_label_encoded)
f1_xgb_raw = f1_score(y_val_encoded, y_pred_xgb_raw, pos_label=pos_label_encoded)

results['XGBoost_Raw'] = {
    'Accuracy': accuracy_xgb_raw,
    'Precision': precision_xgb_raw,
    'Recall': recall_xgb_raw,
    'F1-Score': f1_xgb_raw
}

print("Ham Veri Üzerinde XGBoost Performansı (Doğrulama Seti):")
for metric, value in results['XGBoost_Raw'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# XGBoost modelini PCA verisi üzerinde oluştur ve eğit
xgb_pca = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_pca.fit(X_train_pca, y_train_encoded)

models['XGBoost_PCA'] = xgb_pca

y_pred_xgb_pca = xgb_pca.predict(X_val_pca)

accuracy_xgb_pca = accuracy_score(y_val_encoded, y_pred_xgb_pca)
precision_xgb_pca = precision_score(y_val_encoded, y_pred_xgb_pca, pos_label=pos_label_encoded)
recall_xgb_pca = recall_score(y_val_encoded, y_pred_xgb_pca, pos_label=pos_label_encoded)
f1_xgb_pca = f1_score(y_val_encoded, y_pred_xgb_pca, pos_label=pos_label_encoded)

results['XGBoost_PCA'] = {
    'Accuracy': accuracy_xgb_pca,
    'Precision': precision_xgb_pca,
    'Recall': recall_xgb_pca,
    'F1-Score': f1_xgb_pca
}

print("PCA Verisi Üzerinde XGBoost Performansı (Doğrulama Seti):")
for metric, value in results['XGBoost_PCA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# XGBoost modelini LDA verisi üzerinde oluştur ve eğit
xgb_lda = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_lda.fit(X_train_lda, y_train_encoded)

models['XGBoost_LDA'] = xgb_lda

y_pred_xgb_lda = xgb_lda.predict(X_val_lda)

accuracy_xgb_lda = accuracy_score(y_val_encoded, y_pred_xgb_lda)
precision_xgb_lda = precision_score(y_val_encoded, y_pred_xgb_lda, pos_label=pos_label_encoded)
recall_xgb_lda = recall_score(y_val_encoded, y_pred_xgb_lda, pos_label=pos_label_encoded)
f1_xgb_lda = f1_score(y_val_encoded, y_pred_xgb_lda, pos_label=pos_label_encoded)

results['XGBoost_LDA'] = {
    'Accuracy': accuracy_xgb_lda,
    'Precision': precision_xgb_lda,
    'Recall': recall_xgb_lda,
    'F1-Score': f1_xgb_lda
}

print("LDA Verisi Üzerinde XGBoost Performansı (Doğrulama Seti):")
for metric, value in results['XGBoost_LDA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Gaussian Naive Bayes modelini oluştur ve eğit
gnb_raw = GaussianNB()

gnb_raw.fit(X_train.drop(columns=['Unnamed: 32']), y_train_encoded)

models['GaussianNB_Raw'] = gnb_raw

y_pred_gnb_raw = gnb_raw.predict(X_val.drop(columns=['Unnamed: 32']))

accuracy_gnb_raw = accuracy_score(y_val_encoded, y_pred_gnb_raw)
precision_gnb_raw = precision_score(y_val_encoded, y_pred_gnb_raw, pos_label=pos_label_encoded)
recall_gnb_raw = recall_score(y_val_encoded, y_pred_gnb_raw, pos_label=pos_label_encoded)
f1_gnb_raw = f1_score(y_val_encoded, y_pred_gnb_raw, pos_label=pos_label_encoded)

results['GaussianNB_Raw'] = {
    'Accuracy': accuracy_gnb_raw,
    'Precision': precision_gnb_raw,
    'Recall': recall_gnb_raw,
    'F1-Score': f1_gnb_raw
}

print("Ham Veri Üzerinde Gaussian Naive Bayes Performansı (Doğrulama Seti):")
for metric, value in results['GaussianNB_Raw'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Gaussian Naive Bayes modelini PCA verisi üzerinde oluştur ve eğit
gnb_pca = GaussianNB()
gnb_pca.fit(X_train_pca, y_train_encoded)

models['GaussianNB_PCA'] = gnb_pca
y_pred_gnb_pca = gnb_pca.predict(X_val_pca)

accuracy_gnb_pca = accuracy_score(y_val_encoded, y_pred_gnb_pca)
precision_gnb_pca = precision_score(y_val_encoded, y_pred_gnb_pca, pos_label=pos_label_encoded)
recall_gnb_pca = recall_score(y_val_encoded, y_pred_gnb_pca, pos_label=pos_label_encoded)
f1_gnb_pca = f1_score(y_val_encoded, y_pred_gnb_pca, pos_label=pos_label_encoded)

results['GaussianNB_PCA'] = {
    'Accuracy': accuracy_gnb_pca,
    'Precision': precision_gnb_pca,
    'Recall': recall_gnb_pca,
    'F1-Score': f1_gnb_pca
}

print("PCA Verisi Üzerinde Gaussian Naive Bayes Performansı (Doğrulama Seti):")
for metric, value in results['GaussianNB_PCA'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Gaussian Naive Bayes modelini LDA verisi üzerinde oluştur ve eğit
gnb_lda = GaussianNB()
gnb_lda.fit(X_train_lda, y_train_encoded)

models['GaussianNB_LDA'] = gnb_lda

y_pred_gnb_lda = gnb_lda.predict(X_val_lda)

accuracy_gnb_lda = accuracy_score(y_val_encoded, y_pred_gnb_lda)
precision_gnb_lda = precision_score(y_val_encoded, y_pred_gnb_lda, pos_label=pos_label_encoded)
recall_gnb_lda = recall_score(y_val_encoded, y_pred_gnb_lda, pos_label=pos_label_encoded)
f1_gnb_lda = f1_score(y_val_encoded, y_pred_gnb_lda, pos_label=pos_label_encoded)

results['GaussianNB_LDA'] = {
    'Accuracy': accuracy_gnb_lda,
    'Precision': precision_gnb_lda,
    'Recall': recall_gnb_lda,
    'F1-Score': f1_gnb_lda
}

print("LDA Verisi Üzerinde Gaussian Naive Bayes Performansı (Doğrulama Seti):")
for metric, value in results['GaussianNB_LDA'].items():
    print(f"  {metric}: {value:.4f}")

Soru-8 Validation Performanslarının Ölçülmesi

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).T

results_df.index.name = 'Model'

print("Doğrulama Seti Üzerindeki Model Performans Metrikleri:")
display(results_df.round(4))


In [ ]:
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Ham Veri Üzerinde Lojistik Regresyon ROC-AUC
y_pred_proba_raw_lr = models['LogisticRegression_Raw'].predict_proba(X_val.drop(columns=['Unnamed: 32']))[:, 1]
roc_auc_raw_lr = roc_auc_score(y_val_encoded, y_pred_proba_raw_lr)
results['LogisticRegression_Raw']['ROC-AUC'] = roc_auc_raw_lr

# PCA Verisi Üzerinde Lojistik Regresyon ROC-AUC
y_pred_proba_pca_lr = models['LogisticRegression_PCA'].predict_proba(X_val_pca)[:, 1]
roc_auc_pca_lr = roc_auc_score(y_val_encoded, y_pred_proba_pca_lr)
results['LogisticRegression_PCA']['ROC-AUC'] = roc_auc_pca_lr

# LDA Verisi Üzerinde Lojistik Regresyon ROC-AUC
y_pred_proba_lda_lr = models['LogisticRegression_LDA'].predict_proba(X_val_lda)[:, 1]
roc_auc_lda_lr = roc_auc_score(y_val_encoded, y_pred_proba_lda_lr)
results['LogisticRegression_LDA']['ROC-AUC'] = roc_auc_lda_lr

print("Lojistik Regresyon Modelleri İçin ROC-AUC Skorları Güncellendi.")

In [ ]:
# Ham Veri Üzerinde Random Forest ROC-AUC
y_pred_proba_raw_rf = models['RandomForest_Raw'].predict_proba(X_val.drop(columns=['Unnamed: 32']))[:, 1]
roc_auc_raw_rf = roc_auc_score(y_val_encoded, y_pred_proba_raw_rf)
results['RandomForest_Raw']['ROC-AUC'] = roc_auc_raw_rf

# PCA Verisi Üzerinde Random Forest ROC-AUC
y_pred_proba_pca_rf = models['RandomForest_PCA'].predict_proba(X_val_pca)[:, 1]
roc_auc_pca_rf = roc_auc_score(y_val_encoded, y_pred_proba_pca_rf)
results['RandomForest_PCA']['ROC-AUC'] = roc_auc_pca_rf

# LDA Verisi Üzerinde Random Forest ROC-AUC
y_pred_proba_lda_rf = models['RandomForest_LDA'].predict_proba(X_val_lda)[:, 1]
roc_auc_lda_rf = roc_auc_score(y_val_encoded, y_pred_proba_lda_rf)
results['RandomForest_LDA']['ROC-AUC'] = roc_auc_lda_rf

print("Random Forest Sınıflandırıcısı Modelleri İçin ROC-AUC Skorları Güncellendi.")

In [ ]:
# Ham Veri Üzerinde XGBoost ROC-AUC
y_pred_proba_raw_xgb = models['XGBoost_Raw'].predict_proba(X_val.drop(columns=['Unnamed: 32']))[:, 1]
roc_auc_raw_xgb = roc_auc_score(y_val_encoded, y_pred_proba_raw_xgb)
results['XGBoost_Raw']['ROC-AUC'] = roc_auc_raw_xgb

# PCA Verisi Üzerinde XGBoost ROC-AUC
y_pred_proba_pca_xgb = models['XGBoost_PCA'].predict_proba(X_val_pca)[:, 1]
roc_auc_pca_xgb = roc_auc_score(y_val_encoded, y_pred_proba_pca_xgb)
results['XGBoost_PCA']['ROC-AUC'] = roc_auc_pca_xgb

# LDA Verisi Üzerinde XGBoost ROC-AUC
y_pred_proba_lda_xgb = models['XGBoost_LDA'].predict_proba(X_val_lda)[:, 1]
roc_auc_lda_xgb = roc_auc_score(y_val_encoded, y_pred_proba_lda_xgb)
results['XGBoost_LDA']['ROC-AUC'] = roc_auc_lda_xgb

print("XGBoost Sınıflandırıcısı Modelleri İçin ROC-AUC Skorları Güncellendi.")

In [ ]:
# Ham Veri Üzerinde Gaussian Naive Bayes ROC-AUC
y_pred_proba_raw_gnb = models['GaussianNB_Raw'].predict_proba(X_val.drop(columns=['Unnamed: 32']))[:, 1]
roc_auc_raw_gnb = roc_auc_score(y_val_encoded, y_pred_proba_raw_gnb)
results['GaussianNB_Raw']['ROC-AUC'] = roc_auc_raw_gnb

# PCA Verisi Üzerinde Gaussian Naive Bayes ROC-AUC
y_pred_proba_pca_gnb = models['GaussianNB_PCA'].predict_proba(X_val_pca)[:, 1]
roc_auc_pca_gnb = roc_auc_score(y_val_encoded, y_pred_proba_pca_gnb)
results['GaussianNB_PCA']['ROC-AUC'] = roc_auc_pca_gnb

# LDA Verisi Üzerinde Gaussian Naive Bayes ROC-AUC
y_pred_proba_lda_gnb = models['GaussianNB_LDA'].predict_proba(X_val_lda)[:, 1]
roc_auc_lda_gnb = roc_auc_score(y_val_encoded, y_pred_proba_lda_gnb)
results['GaussianNB_LDA']['ROC-AUC'] = roc_auc_lda_gnb

print("Gaussian Naive Bayes Modelleri İçin ROC-AUC Skorları Güncellendi.")

### Güncel Model Performans Karşılaştırması (Doğrulama Seti)

In [ ]:
import pandas as pd

# Sonuçları DataFrame'e dönüştürün
results_df = pd.DataFrame(results).T

# Sütun isimlerini daha okunabilir hale getirin
results_df.index.name = 'Model'

print("Doğrulama Seti Üzerindeki Güncel Model Performans Metrikleri:")
display(results_df.round(4))

In [ ]:
# LDA dönüşümünü test setine uygula
X_test_lda = lda.transform(X_test.drop(columns=['Unnamed: 32']))

y_test_encoded = le.transform(y_test)

# XGBoost_LDA modelini kullanarak test seti üzerinde tahminler
y_pred_xgb_lda_test = models['XGBoost_LDA'].predict(X_test_lda)
y_pred_proba_xgb_lda_test = models['XGBoost_LDA'].predict_proba(X_test_lda)[:, 1]

# Performans metriklerini hesapla
accuracy_xgb_lda_test = accuracy_score(y_test_encoded, y_pred_xgb_lda_test)
precision_xgb_lda_test = precision_score(y_test_encoded, y_pred_xgb_lda_test, pos_label=pos_label_encoded)
recall_xgb_lda_test = recall_score(y_test_encoded, y_pred_xgb_lda_test, pos_label=pos_label_encoded)
f1_xgb_lda_test = f1_score(y_test_encoded, y_pred_xgb_lda_test, pos_label=pos_label_encoded)
roc_auc_xgb_lda_test = roc_auc_score(y_test_encoded, y_pred_proba_xgb_lda_test)

test_results = {
    'XGBoost_LDA_Test': {
        'Accuracy': accuracy_xgb_lda_test,
        'Precision': precision_xgb_lda_test,
        'Recall': recall_xgb_lda_test,
        'F1-Score': f1_xgb_lda_test,
        'ROC-AUC': roc_auc_xgb_lda_test
    }
}

print("XGBoost_LDA Modelinin Test Seti Performansı:")
for metric, value in test_results['XGBoost_LDA_Test'].items():
    print(f"  {metric}: {value:.4f}")

### XGBoost_LDA Modelinin Test Seti Karışıklık Matrisi (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test_encoded, y_pred_xgb_lda_test)

# Sınıf etiketleri (le.classes_ kullanılarak orijinal etiketler alınabilir)
class_labels = le.classes_

# Confusion Matrix'i görselleştirme
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('XGBoost_LDA Model - Test Seti Karışıklık Matrisi')
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.show()

### XGBoost_LDA Modelinin Test Seti ROC Eğrisi ve AUC Değeri

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Tahmin olasılıklarını al
y_pred_proba_xgb_lda_test = models['XGBoost_LDA'].predict_proba(X_test_lda)[:, 1]

# ROC eğrisi için FPR, TPR ve eşik değerlerini hesapla
fpr, tpr, thresholds = roc_curve(y_test_encoded, y_pred_proba_xgb_lda_test)

# AUC değerini hesapla
auc_score = roc_auc_score(y_test_encoded, y_pred_proba_xgb_lda_test)

# ROC eğrisini çiz
plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Rastgele Sınıflandırıcı')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Yanlış Pozitif Oranı (False Positive Rate)')
plt.ylabel('Doğru Pozitif Oranı (True Positive Rate)')
plt.title('XGBoost_LDA Model - Test Seti ROC Eğrisi')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"XGBoost_LDA Modelinin Test Seti Üzerindeki AUC Değeri: {auc_score:.4f}")

In [ ]:
!pip install -q shap

In [ ]:
import shap
import numpy as np

# En iyi modelimiz XGBoost_LDA idi. Bu modeli yükle
best_model = models['XGBoost_LDA']

explainer = shap.TreeExplainer(best_model)

shap_values = explainer.shap_values(X_test_lda)

print("\nXGBoost_LDA Modelinin Özellik Önem Derecesi ve Yönü (SHAP Summary Plot):")
shap.summary_plot(shap_values, X_test_lda, feature_names=[f'LDA_{i+1}' for i in range(X_test_lda.shape[1])], class_names=le.classes_)

In [ ]:
print("\nXGBoost_LDA Modelinin Özellik Önem Derecesi (SHAP Bar Plot):")
shap.summary_plot(shap_values, X_test_lda, feature_names=[f'LDA_{i+1}' for i in range(X_test_lda.shape[1])], plot_type="bar", class_names=le.classes_)


In [ ]:
import shap
import numpy as np

# En iyi PCA modelimiz XGBoost_PCA idi. Bu modeli yükle
best_pca_model = models['XGBoost_PCA']

explainer_pca = shap.TreeExplainer(best_pca_model)

X_test_pca = pca_final.transform(X_test.drop(columns=['Unnamed: 32']))

shap_values_pca = explainer_pca.shap_values(X_test_pca)

print("\nXGBoost_PCA Modelinin Özellik Önem Derecesi ve Yönü (SHAP Summary Plot):")
shap.summary_plot(shap_values_pca, X_test_pca, feature_names=[f'PC_{i+1}' for i in range(X_test_pca.shape[1])], class_names=le.classes_)

In [ ]:
print("\nXGBoost_PCA Modelinin Özellik Önem Derecesi (SHAP Bar Plot):")
shap.summary_plot(shap_values_pca, X_test_pca, feature_names=[f'PC_{i+1}' for i in range(X_test_pca.shape[1])], plot_type="bar", class_names=le.classes_)